# Chapter 2 — Worked Example: Preparing the Hennaya Piezometric Dataset

**AI for Hydrogeologists** — companion notebook

This notebook reproduces the complete data preparation workflow of Section 2.5:
loading raw piezometric campaigns (1981, 2012, 2022) from the Hennaya plain
(northwestern Algeria), quality control, and enrichment with precipitation,
elevation, and land cover covariates.

Data files required in the same folder (or upload them to the Colab session):
`head_1981.txt`, `head_2012.txt`, `head_2022.txt`, `Précipitations_mensuelles.xlsx`,
`wells_dem_landcover.csv` (pre-computed in Step 3, see note below).


In [ ]:
!pip install pyproj -q
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)


## Step 1 — Load and clean the three piezometric campaigns

In [ ]:
FILES = {
    1981: "head_1981.txt",
    2012: "head_2012.txt",
    2022: "head_2022.txt",
}

frames = []
for year, path in FILES.items():
    df = pd.read_csv(path, sep="\t")
    df = df.rename(columns={
        "Well Name": "field_label",
        "X [m]": "x", "Y [m]": "y",
        "Screen Elev. [m]": "screen_elev_m",
        "HEAD [m]": "head_m",
        "Obs. Time [day]": "obs_time_day",
    })
    df["campaign_year"] = year
    # Field labels (e.g. "P1") are re-used independently in each campaign and do
    # NOT refer to the same physical well across years (confirmed with the data
    # owner). We therefore assign new, unambiguous, campaign-specific IDs.
    df = df.sort_values(["y", "x"]).reset_index(drop=True)
    df["well_id"] = [f"HYN-{year}-{i+1:03d}" for i in range(len(df))]
    frames.append(df[["well_id", "field_label", "campaign_year", "x", "y",
                       "screen_elev_m", "head_m"]])

heads = pd.concat(frames, ignore_index=True)
print("Records per campaign:")
print(heads.groupby("campaign_year").size())


In [ ]:
# Physical sanity check: HEAD should generally exceed Screen Elev.
heads["head_minus_screen"] = heads["head_m"] - heads["screen_elev_m"]
print(heads.groupby("campaign_year")["head_minus_screen"].describe()[["min","mean","max"]])

flagged = heads[heads["head_minus_screen"] < -10]
print(f"\nRecords with a large negative HEAD - Screen Elev. gap (data quality flag): {len(flagged)}")
flagged[["well_id","field_label","campaign_year","head_m","screen_elev_m"]]


**Note on `HYN-2022-033` (field label P139):** a gap of -58.6 m is physically
implausible compared with neighbouring wells and is excluded from spatial
analyses in Chapter 4 (kept here, flagged, for the data-cleaning illustration).

## Step 2 — Precipitation covariates (Zenata station, 1980/81-2024/25)

In [ ]:
precip = pd.read_excel("Précipitations_mensuelles.xlsx")
months = ["Sep","Oct","Nov","Dec","Jan","Fev","Mars","Avr","Mai","Juin","Juill","Aout"]
precip["P_annual_mm"] = precip[months].sum(axis=1)
precip["hydro_year_start"] = precip["Année_hydro"].str.split("/").str[0].astype(int)

# 1980/81 only has Sep-Dec populated -> incomplete, not a true zero-rainfall year
precip.loc[precip["hydro_year_start"] == 1980, "P_annual_mm"] = np.nan

mean_annual = precip["P_annual_mm"].mean()
print(f"Mean annual precipitation at Zenata (excl. incomplete 1980/81): {mean_annual:.1f} mm")

def covariates_for_year(census_year, n_hist=(1, 3, 5)):
    out = {"campaign_year": census_year}
    ref = precip[precip["hydro_year_start"] == census_year - 1]
    out["P_year_of_survey_mm"] = ref["P_annual_mm"].values[0] if len(ref) else np.nan
    for n in n_hist:
        window = precip[(precip["hydro_year_start"] >= census_year - n) &
                         (precip["hydro_year_start"] <= census_year - 1)]
        out[f"P_cumul_{n}yr_mm"] = window["P_annual_mm"].sum()
        out[f"P_anomaly_{n}yr_pct"] = 100 * (window["P_annual_mm"].mean() - mean_annual) / mean_annual
    return out

precip_cov = pd.DataFrame([covariates_for_year(y) for y in [1981, 2012, 2022]])
precip_cov


**Note:** the precipitation record begins in 1980/81, exactly at the first
census. No pre-survey climatic context is therefore available for the 1981
baseline; covariates are only meaningful for the 2012 and 2022 campaigns.

## Step 3 — Elevation and land cover (DEM + ESA WorldCover)

This step needs live internet access (SRTM via Open Topo Data, ESA WorldCover
via Microsoft Planetary Computer STAC) and is written as a **separate cell**
so it only needs to run once; its output (`wells_dem_landcover.csv`) is
re-used below. If that file is already in your session, skip straight to
Step 4.

In [ ]:
# Run this cell only if wells_dem_landcover.csv is not already available.
import os
if not os.path.exists("wells_dem_landcover.csv"):
    !pip install rasterio rioxarray pystac-client planetary-computer -q
    import requests, time
    from pyproj import Transformer

    wells = heads[["well_id", "x", "y"]].drop_duplicates("well_id").reset_index(drop=True)
    transformer = Transformer.from_crs("EPSG:32630", "EPSG:4326", always_xy=True)  # UTM 30N, confirmed for Hennaya
    lon, lat = transformer.transform(wells["x"].values, wells["y"].values)
    wells["lon"], wells["lat"] = lon, lat

    def fetch_elevation_batch(lats, lons, batch_size=90, pause=1.0):
        elevations = []
        for i in range(0, len(lats), batch_size):
            locs = "|".join(f"{la},{lo}" for la, lo in zip(lats[i:i+batch_size], lons[i:i+batch_size]))
            r = requests.get(f"https://api.opentopodata.org/v1/srtm30m?locations={locs}", timeout=30)
            r.raise_for_status()
            elevations.extend([res["elevation"] for res in r.json()["results"]])
            time.sleep(pause)
        return elevations

    wells["elevation_srtm_m"] = fetch_elevation_batch(wells["lat"].tolist(), wells["lon"].tolist())

    import pystac_client, planetary_computer, rioxarray
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace)
    bbox = [wells["lon"].min()-0.02, wells["lat"].min()-0.02,
            wells["lon"].max()+0.02, wells["lat"].max()+0.02]
    items = list(catalog.search(collections=["esa-worldcover"], bbox=bbox).items())

    WORLDCOVER_CLASSES = {10:"Tree cover",20:"Shrubland",30:"Grassland",40:"Cropland",
                           50:"Built-up",60:"Bare/sparse vegetation",70:"Snow and ice",
                           80:"Permanent water bodies",90:"Herbaceous wetland",
                           95:"Mangroves",100:"Moss and lichen"}
    da = rioxarray.open_rasterio(items[0].assets["map"].href).squeeze()
    def sample_lc(lon, lat):
        ix = np.abs(da.x.values - lon).argmin(); iy = np.abs(da.y.values - lat).argmin()
        code = int(da.values[iy, ix]); return code, WORLDCOVER_CLASSES.get(code, "Unknown")
    codes, labels = zip(*[sample_lc(lo, la) for lo, la in zip(wells["lon"], wells["lat"])])
    wells["landcover_code"], wells["landcover_class"] = codes, labels
    wells.to_csv("wells_dem_landcover.csv", index=False)
    print("Fetched and saved wells_dem_landcover.csv")
else:
    print("wells_dem_landcover.csv already present, skipping live fetch.")


## Step 3b — Soil texture (SoilGrids, substitute for the too-coarse GLiM)

The global lithological map (GLiM) is only freely available at 0.5 resolution
(about 55 km at this latitude) - far too coarse to distinguish anything within
the ~5 km wide Hennaya plain; every well would receive the identical value.
We use SoilGrids v2.0 (250 m, ISRIC) instead, which resolves real local
variation in near-surface texture. See `06_colab_soilgrids.py` (fair-use rate
limited, takes several minutes to run) for the fetching code.

## Step 4 — Merge everything into one ML-ready table

In [ ]:
dem = pd.read_csv("wells_dem_landcover_soil.csv")

df = heads.merge(
    dem[["well_id","elevation_srtm_m","landcover_code","landcover_class",
         "clay_pct","sand_pct","silt_pct","texture_class"]],
    on="well_id", how="left")
df = df.merge(precip_cov, on="campaign_year", how="left")

print("Missing soil texture (mostly built-up land, no SoilGrids prediction "
      "over artificial surfaces):")
print(df[df["clay_pct"].isna()]["landcover_class"].value_counts())

# Engineered features
df["depth_to_water_approx_m"] = df["elevation_srtm_m"] - df["head_m"]
df["is_cropland"] = (df["landcover_code"] == 40).astype(int)
df["qc_flag_head_below_screen"] = (df["head_m"] < df["screen_elev_m"]).astype(int)

print("Final shape:", df.shape)
df.to_csv("hennaya_heads_ml_ready.csv", index=False)
df.head()


### A cautionary note on naive comparison across campaigns

The raw per-campaign mean of `depth_to_water_approx_m` **increases** from 1981
to 2022. This does **not** by itself indicate aquifer depletion: each campaign
samples a different, only partially overlapping set of locations (see Chapter
3, Section 3.5.1, on why naive comparisons across non-exchangeable samples are
misleading). The spatially honest trend analysis is carried out separately in
the Chapter 4 case study notebook, which interpolates each campaign onto a
common grid before differencing.